<div dir="rtl" lang="he" align="right" markdown="1">

# RAG בעברית

בפרק הקודם בנינו `pipeline` שלם של `retrieval` ומדדנו אותו. עכשיו נריץ אותו על עברית, ונראה מה נשבר.

שלושה דברים מיוחדים לעברית ומשפיעים ישירות על החיפוש. היא מדביקה מילות תפקוד לראש המילה הבאה, היא נכתבת בלי ניקוד, והיא נחשבת יקרה ב-`tokens`, הדבר האחרון יתברר כנכון רק בחצי מהמקרים.

**המדידה כאן על נתונים אמיתיים:** 150 קטעים ו-1,036 שאלות שנכתבו בידי בני אדם, מתוך `HeQ`, קטעים מוויקיפדיה ומגיקטיים.

**ואזהרה מראש על מסקנה אחת:** המודל שבחרנו בפרק 01 הוא הגרוע מבין ארבעה שנבדקו על עברית. הפרק הזה מוכיח את זה ומחליף אותו.

</div>

In [ ]:
from pathlib import Path

from aihe import viz
from aihe.hebrew import (
    compare_to_reference,
    load_reference,
    normalize,
    strip_prefixes,
    tokenize_hebrew,
)
from aihe.pipeline import (
    encode_corpus,
    evaluate,
    keyword_retriever,
    load_corpus,
    passage_retriever,
)
from aihe.retrieval import tokenize

HERE = Path("data") if Path("data/heq-subset.json").exists() else Path("chapters/02-hebrew-rag/data")
corpus = load_corpus(HERE / "heq-subset.json")
print(len(corpus.documents), "passages,", len(corpus.queries), "questions")
print(corpus.documents[0]["title"])

<div dir="rtl" lang="he" align="right" markdown="1">

## מה שעברית מדביקה

המילה `ובמסמכים` היא מילה אחת לכל `tokenizer` בעולם, ולקורא היא שלושה חלקים: `ו` ועוד `ב` ועוד `מסמכים`. זה אומר ששאילתה על `מסמכים` לא תמצא מסמך שכתוב בו `ובמסמכים`, כי אין ביניהם שום תו משותף מבחינת החיפוש.

באנגלית הבעיה הזו כמעט לא קיימת: המילים הקטנות עומדות בנפרד. בעברית הן חלק מהמילה.

</div>

In [ ]:
words = ["ובמסמכים", "שבמאגר", "כשהטוקן", "להצפנה", "השרתים", "מידע", "שולחן"]
for w in words:
    print(f"  {w:10s} -> {'+'.join(strip_prefixes(w))}")

print("\nניקוד:", repr(normalize("בְּרֵאשִׁית")), "==", repr(normalize("בראשית")))

<div dir="rtl" lang="he" align="right" markdown="1">

## כמה עברית באמת עולה

האמירה שעברית יקרה ב-`tokens` נכונה, אבל רק אם ה-`tokenizer` נבנה לאנגלית. זו בחירה של כלי ולא תכונה של השפה, וההבדל גדול בהרבה ממה שרוב האנשים מנחשים.

התא הבא לוקח את אותה מילה בדיוק ומראה מה כל אחד משני ה-`tokenizers` עושה איתה.

</div>

In [ ]:
from transformers import AutoTokenizer

gpt2 = AutoTokenizer.from_pretrained("gpt2")
xlmr = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")

word = "ובמסמכים"
g = gpt2.convert_ids_to_tokens(gpt2.encode(word))
x = xlmr.convert_ids_to_tokens(xlmr.encode(word, add_special_tokens=False))
print(f"GPT-2  ({len(g):2d} tokens): {g}")
print(f"XLM-R  ({len(x):2d} tokens): {x}")

In [ ]:
PAIRS = [
    ("The access token is valid for sixty minutes from the moment it is issued.",
     "טוקן הגישה תקף למשך שישים דקות מרגע שהונפק."),
    ("Every request carries a bearer token in the authorization header.",
     "כל בקשה נושאת טוקן בכותרת ההרשאה."),
    ("Records written to a project never leave the region they were created in.",
     "רשומות שנכתבות לפרויקט לעולם לא עוזבות את האזור שבו נוצרו."),
    ("Connections require TLS and plain requests are refused rather than redirected.",
     "החיבורים דורשים הצפנה ובקשות רגילות נדחות במקום להיות מנותבות."),
]
for tok, label in ((gpt2, "GPT-2 (English BPE)"), (xlmr, "XLM-R (multilingual)")):
    en = sum(len(tok.encode(e, add_special_tokens=False)) for e, _ in PAIRS)
    he = sum(len(tok.encode(h, add_special_tokens=False)) for _, h in PAIRS)
    print(f"  {label:24s} English {en:4d}   Hebrew {he:4d}   ratio {he / en:.2f}x")

<div dir="rtl" lang="he" align="right" markdown="1">

## לפצל את התחיליות: כלל פשוט מול מודל

יש שתי דרכים לפצל את התחיליות. אפשר לכתוב כלל: אם המילה מתחילה באחת מאותיות התפקוד ונשאר גזע ארוך מספיק, מפרידים. זה עולה חמש-עשרה שורות ואפס הורדות.

אפשר גם להשתמש במודל ייעודי, `dictabert-seg`, שמדויק יותר ושוקל כ-700 מגה. הרצנו אותו פעם אחת על הטקסט של הפרק ושמרנו את התוצאה, כדי שאפשר יהיה להשוות בלי להוריד אותו.

**שימו לב לפני שתשתמשו בו בעצמכם:** המודל הזה נטען עם ראש אקראי בגרסאות `transformers` הנוכחיות, לא זורק שום שגיאה, ומפצל בביטחון גמור לא נכון. `PREFLIGHT.md` מסביר.

</div>

In [ ]:
reference = load_reference(HERE / "dictabert-seg.json")
result = compare_to_reference(reference)
print(f"{result['total']} מילים: הכלל מסכים עם המודל ב-{result['rate']:.1%}\n")

print(f"  {'מילה':14s} {'המודל':18s} {'הכלל':18s}")
for word, model_says, rule_says in result["disagreements"][:8]:
    print(f"  {word:14s} {'+'.join(model_says):18s} {'+'.join(rule_says):18s}")

<div dir="rtl" lang="he" align="right" markdown="1">

## והאם זה בכלל עוזר

עכשיו המדידה. נריץ `BM25` שלוש פעמים על אותם 150 קטעים ואותן 1,036 שאלות, ונשנה רק את ה-`tokenizer`: פעם אחת כמו בפרק 01, פעם אחת עם נרמול ניקוד בלבד, ופעם אחת עם פיצול התחיליות.

</div>

In [ ]:
runs = {
    "chapter 01 tokenizer": keyword_retriever(corpus, tokenize),
    "+ normalise only": keyword_retriever(corpus, lambda t: tokenize_hebrew(t, split=False)),
    "+ split prefixes": keyword_retriever(corpus, tokenize_hebrew),
}
scores = {name: evaluate(r, corpus, k=5) for name, r in runs.items()}

print(f"{'setup':24s} {'MRR':>7s} {'r@1':>7s} {'r@5':>7s} {'failed@5':>9s}")
for name, s in scores.items():
    r1 = evaluate(runs[name], corpus, k=1)["recall@1"]
    print(f"{name:24s} {s['MRR']:7.3f} {r1:7.3f} {s['recall@5']:7.3f} {s['failed@5']:9.3f}")

In [ ]:
viz.climb({name: s["MRR"] for name, s in scores.items()}, label="MRR",
          title="Hebrew BM25: only the tokenizer changes");

<div dir="rtl" lang="he" align="right" markdown="1">

## למה כלל שטועה ברבע מהמקרים עדיין עוזר

זו הנקודה המעניינת בפרק. הכלל מסכים עם המודל רק בשלושה רבעים מהמילים, ובכל זאת הוא משפר את התוצאה.

הסיבה היא בתכנון ולא במזל. אנחנו מוסיפים את הגזע לאינדקס ולא מחליפים בו את המילה המקורית. גזע שגוי מוסיף `token` מיותר שאף שאילתה לא תבקש, כלומר רעש, אבל הוא לעולם לא מוחק התאמה שהייתה נכונה.

כשמתכננים את אופן הכישלון מראש, אפשר להרשות לעצמך רכיב גרוע יותר. אם היינו מחליפים את המילה בגזע, אותו כלל בדיוק היה פוגע במקום לעזור.

</div>

<div dir="rtl" lang="he" align="right" markdown="1">

## והמודל מפרק 01

נשארה שאלה אחת. בפרק 01 בחרנו מודל `embedding` קטן ורב-לשוני, והשאלה היא כמה הוא טוב על עברית.

נשווה אותו למודל רב-לשוני אחר באותו סדר גודל. שימו לב שהמודל השני הוא **א-סימטרי**: הוא אומן עם הקידומות `query:` ו-`passage:`, והשמטתן עולה בדיוק בלי להודיע על כך בשום צורה. זו מלכודת שקל מאוד ליפול בה.

</div>

In [ ]:
from sentence_transformers import SentenceTransformer

results = {}
for name, asymmetric in [
    ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", False),
    ("intfloat/multilingual-e5-small", True),
]:
    model = SentenceTransformer(name, device="cpu")
    docs, questions = encode_corpus(model, corpus, asymmetric=asymmetric)
    scored = evaluate(passage_retriever(questions, docs, corpus), corpus, k=5)
    results[name.split("/")[-1]] = scored
    print(f"  {name.split('/')[-1][:38]:40s} MRR={scored['MRR']:.3f}  r@5={scored['recall@5']:.3f}")

In [ ]:
# the same model, with its prefixes left off - no error, just a worse number
model = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
docs, questions = encode_corpus(model, corpus, asymmetric=False)
without = evaluate(passage_retriever(questions, docs, corpus), corpus, k=5)
with_prefixes = results["multilingual-e5-small"]

print(f"  with    query:/passage:   MRR={with_prefixes['MRR']:.3f}")
print(f"  without query:/passage:   MRR={without['MRR']:.3f}")
print(f"  the prefixes are worth {with_prefixes['MRR'] - without['MRR']:+.3f} MRR")

<div dir="rtl" lang="he" align="right" markdown="1">

## מה שהמדידה אומרת, כולל מה שלא נעים בה

שלוש מסקנות, והשלישית היא זו שצריך לזכור.

הראשונה, `BM25` עם פיצול תחיליות הוא בסיס חזק מאוד בעברית, והוא עולה אפס. במדידה המלאה שלנו הוא ניצח שלושה מתוך ארבעה מודלים שנבדקו.

השנייה, המודל מפרק 01 באמת חלש על עברית, והמודל הרב-לשוני השני טוב ממנו בבירור. מודלים שאומנו במיוחד לעברית, שגם אותם בדקנו, הפסידו לשניהם, כי מה שזמין מהם כ-`embedding` הוא ישן או קטן יותר. ההנחה ש"לעברית צריך מודל עברי" לא החזיקה במדידה.

השלישית היא הסתייגות על המספרים של עצמנו. `HeQ` הוא `extractive`: כל תשובה היא קטע מתוך הפסקה, ולכן השאלות חולקות מילים עם הפסקה שלהן מעצם הבנייה. זה מטיב עם חיפוש מילולי. `BM25` באמת חזק כאן, אבל חלק מהחוזק הזה שייך למדד ולא לשיטה, ולכן מדדו על הנתונים שלכם.

</div>

In [ ]:
best_bm25 = scores["+ split prefixes"]
baseline = scores["chapter 01 tokenizer"]
print(f"tokenizer alone: MRR {baseline['MRR']:.3f} -> {best_bm25['MRR']:.3f}"
      f"   failed@5 {baseline['failed@5']:.1%} -> {best_bm25['failed@5']:.1%}")
print(f"best embedder here: MRR {max(s['MRR'] for s in results.values()):.3f}")